# BBKNN Batch Integration Pipeline

**BBKNN (Batch Balanced k-Nearest Neighbors) Integration**

- Graph-based fast batch integration
- Highly variable gene selection
- Batch integration quality assessment
- Multi-resolution clustering
- Enhanced visualization

**Author:** Clinical-Bioinformatics Team  
**Date:** 2025-11-01  
**Version:** v1.0

## 1. Import Libraries and Setup

In [ ]:
import sys
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import time
from datetime import datetime
import json

import scanpy as sc
import bbknn
import matplotlib.pyplot as plt
from scipy.stats import entropy
from sklearn.metrics import silhouette_score

warnings.filterwarnings('ignore')

# Check versions
print(f"scanpy: {sc.__version__}")
print(f"Python: {sys.version}")
print(f"NumPy: {np.__version__}")
print(f"Pandas: {pd.__version__}")

## 2. Configuration Parameters

In [ ]:
# ========== Input/Output Configuration ==========
INPUT_H5AD_PATH = "/home/h2048/data/R/1124/merged_object_sc.h5ad"
OUTPUT_DIR = "/home/h2048/data/py/1125/bbknn_output_optimized"
OVERWRITE_EXISTING = True

# ========== BBKNN Integration Configuration ==========
BATCH_KEY = "dataset"  # Batch variable name

# BBKNN core parameters
BBKNN_NEIGHBORS_WITHIN_BATCH = 3  # Number of neighbors within each batch (recommended 3-5)
BBKNN_N_PCS = 50  # Number of principal components to use

# ========== Highly Variable Genes Selection ==========
USE_HVG = True  # Whether to select highly variable genes
N_TOP_GENES = 4000  # Number of highly variable genes to keep
HVG_FLAVOR = "seurat_v3"  # HVG selection method

# ========== Gene Filtering Configuration ==========
MIN_CELLS_PER_GENE = 3

# ========== Normalization Configuration ==========
NORMALIZE_TOTAL = True  # Whether to perform total count normalization
TARGET_SUM = 1e4  # Target sum for normalization
LOG_TRANSFORM = True  # Whether to perform log transformation
SCALE_DATA = True  # Whether to scale data
MAX_VALUE = 10  # Maximum value for scaling

# ========== PCA Configuration ==========
N_PCS = 50  # Number of principal components to compute

# ========== Dimensionality Reduction and Visualization ==========
RUN_UMAP = True
UMAP_MIN_DIST = 0.5
UMAP_N_NEIGHBORS = 100  # UMAP neighbors (will be overridden by BBKNN neighbor graph)

# ========== Clustering Configuration (Multi-resolution) ==========
RUN_CLUSTERING = True
LEIDEN_RESOLUTIONS = [1.0, 1.4, 1.8, 2.2]  # Multiple resolutions
DEFAULT_RESOLUTION = 1.4  # Default resolution to use

# ========== Batch Integration Evaluation ==========
EVALUATE_INTEGRATION = False  # Whether to evaluate batch integration quality

# ========== Visualization Variables ==========
VISUALIZATION_VARS = [
    "dataset",
    "Annotation",
    "tissue_sampling_method",
]

# ========== Enhanced Visualization ==========
GENERATE_FACET_PLOTS = True  # Whether to generate faceted plots by batch

VERBOSE = True

print("Configuration loaded successfully")
print(f"Input file: {INPUT_H5AD_PATH}")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Batch key: {BATCH_KEY}")
print(f"BBKNN neighbors within batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")

## 3. Create Output Directory

In [ ]:
# Create output directory
output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

fig_dir = output_dir / "figures"
fig_dir.mkdir(exist_ok=True)

print(f"Output directory created: {output_dir}")
print(f"Figures directory: {fig_dir}")

# Set scanpy figure directory
sc.settings.figdir = fig_dir

# Record start time
start_time = time.time()
print(f"\nAnalysis started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 4. Load Data

In [46]:
print("="*70)
print("Step 1: Loading Data")
print("="*70)

print(f"\nReading file: {INPUT_H5AD_PATH}")

if not Path(INPUT_H5AD_PATH).exists():
    raise FileNotFoundError(f"File not found: {INPUT_H5AD_PATH}")

# Read data
adata = sc.read_h5ad(INPUT_H5AD_PATH)

print(f"Data loaded successfully")
print(f"   Cells: {adata.n_obs:,}")
print(f"   Genes: {adata.n_vars:,}")

# Display obs columns
print(f"\nAvailable metadata columns:")
for col in adata.obs.columns:
    n_unique = adata.obs[col].nunique()
    print(f"   - {col}: {n_unique} unique values")

# Check batch key
if BATCH_KEY not in adata.obs.columns:
    raise ValueError(f"Batch key '{BATCH_KEY}' not found in adata.obs")

# Display batch distribution
print(f"\nBatch distribution (key: {BATCH_KEY}):")
batch_counts = adata.obs[BATCH_KEY].value_counts().sort_index()
for batch, count in batch_counts.items():
    pct = count / adata.n_obs * 100
    print(f"   {batch}: {count:,} cells ({pct:.1f}%)")

Data loaded successfully
   Cells: 456,356
   Genes: 58,184

Available metadata columns:
   - orig.ident: 32 unique values
   - nCount_RNA: 58304 unique values
   - nFeature_RNA: 5800 unique values
   - plateID: 29 unique values
   - status: 2 unique values
   - donorID: 7 unique values
   - cDate: 7 unique values
   - age: 7 unique values
   - sex: 5 unique values
   - cellType: 26 unique values
   - percent.mt: 348184 unique values
   - percent.ribo: 291526 unique values
   - tissue: 4 unique values
   - tissue_sampling_method: 5 unique values
   - dataset: 23 unique values
   - sample: 165 unique values
   - percent.rb: 82362 unique values
   - decontX_contamination: 456356 unique values
   - decontX_clusters: 300 unique values
   - nCount_decontXcounts: 27364 unique values
   - nFeature_decontXcounts: 5863 unique values
   - donor_id: 96 unique values
   - Group: 3 unique values
   - Ethnicity_inferred: 6 unique values
   - Smoker: 3 unique values
   - COVID_status: 2 unique values

## 5. Preprocessing for BBKNN

In [47]:
print("\n" + "="*70)
print("Step 2: Preprocessing for BBKNN")
print("="*70)

# Save original counts if not already saved
if 'counts' not in adata.layers:
    print("\nSaving raw counts to layers['counts']...")
    adata.layers['counts'] = adata.X.copy()
else:
    print("\nRaw counts already saved in layers['counts']")


Step 2: Preprocessing for BBKNN

Saving raw counts to layers['counts']...


In [48]:
# Gene filtering
print(f"\nGene filtering (min_cells={MIN_CELLS_PER_GENE})...")
n_genes_before = adata.n_vars
sc.pp.filter_genes(adata, min_cells=MIN_CELLS_PER_GENE)
n_genes_after = adata.n_vars

print(f"   Genes before: {n_genes_before:,}")
print(f"   Genes after: {n_genes_after:,}")
print(f"   Removed: {n_genes_before - n_genes_after:,}")


Gene filtering (min_cells=3)...
   Genes before: 58,184
   Genes after: 58,184
   Removed: 0


In [49]:
# Normalization
if NORMALIZE_TOTAL:
    print(f"\nNormalizing total counts (target_sum={TARGET_SUM})...")
    sc.pp.normalize_total(adata, target_sum=TARGET_SUM)
    print("   Normalization completed")


Normalizing total counts (target_sum=10000.0)...
   Normalization completed


In [50]:
# Log transformation
if LOG_TRANSFORM:
    print("\nApplying log1p transformation...")
    sc.pp.log1p(adata)
    print("   Log transformation completed")


Applying log1p transformation...
   Log transformation completed


In [51]:
# Highly variable genes selection
if USE_HVG:
    print(f"\nSelecting highly variable genes (n={N_TOP_GENES})...")
    print(f"   Method: {HVG_FLAVOR}")
    
    sc.pp.highly_variable_genes(
        adata,
        n_top_genes=N_TOP_GENES,
        flavor=HVG_FLAVOR,
        batch_key=BATCH_KEY,
        subset=False  # Don't subset yet, keep all gene info
    )
    
    n_hvg = adata.var['highly_variable'].sum()
    print(f"   Selected {n_hvg} highly variable genes")
    
    # Create subset for downstream analysis
    adata_hvg = adata[:, adata.var['highly_variable']].copy()
    print(f"   Final genes for downstream analysis: {adata_hvg.n_vars:,}")
else:
    adata_hvg = adata.copy()
    print("\nSkipping HVG selection, using all genes")


Selecting highly variable genes (n=4000)...
   Method: seurat_v3
   Selected 4000 highly variable genes
   Final genes for downstream analysis: 4,000


In [52]:
# Scale data (only on HVG)
if SCALE_DATA:
    print(f"\nScaling data (max_value={MAX_VALUE})...")
    sc.pp.scale(adata_hvg, max_value=MAX_VALUE)
    print("   Scaling completed")

print("\nPreprocessing completed")
print(f"   Final dimensions: {adata_hvg.n_obs:,} cells x {adata_hvg.n_vars:,} genes")


Scaling data (max_value=10)...
   Scaling completed

Preprocessing completed
   Final dimensions: 456,356 cells x 4,000 genes


## 6. Analyze Unintegrated Data (Baseline)

In [53]:
print("\n" + "="*70)
print("Step 3: Analyzing Unintegrated Data (Baseline)")
print("="*70)

# Create temporary copy for unintegrated analysis
adata_temp = adata.copy()

# Use preprocessed data
if 'counts' in adata_temp.layers:
    print("\nUsing preprocessed data...")
    adata_temp.X = adata_temp.layers['counts'].copy()
    
    # Normalize
    if NORMALIZE_TOTAL:
        sc.pp.normalize_total(adata_temp, target_sum=TARGET_SUM)
    if LOG_TRANSFORM:
        sc.pp.log1p(adata_temp)

# If using HVG, subset to highly variable genes
if USE_HVG and 'highly_variable' in adata_temp.var:
    print("Subsetting to highly variable genes...")
    adata_temp = adata_temp[:, adata_temp.var['highly_variable']].copy()

# Scale and PCA
print("\nRunning PCA on unintegrated data...")
if SCALE_DATA:
    sc.pp.scale(adata_temp, max_value=MAX_VALUE)
sc.tl.pca(adata_temp, n_comps=50)

# Standard neighbors and UMAP (without BBKNN)
print("Computing standard neighbors and UMAP...")
sc.pp.neighbors(adata_temp, n_neighbors=UMAP_N_NEIGHBORS, use_rep="X_pca")
sc.tl.umap(adata_temp, min_dist=UMAP_MIN_DIST)

print("   Unintegrated analysis completed")


Step 3: Analyzing Unintegrated Data (Baseline)

Using preprocessed data...
Subsetting to highly variable genes...

Running PCA on unintegrated data...
Computing standard neighbors and UMAP...
   Unintegrated analysis completed


In [54]:
# Visualize unintegrated data
print("\nGenerating unintegrated visualizations...")

# UMAP by batch
if BATCH_KEY in adata_temp.obs.columns:
    sc.pl.umap(
        adata_temp,
        color=BATCH_KEY,
        show=False,
        title='UMAP - Unintegrated (by batch)',
        save='_unintegrated_batch.png'
    )
    print(f"   Saved: {fig_dir}/umap_unintegrated_batch.png")

# UMAP by cell type (if available)
if 'cell_type' in adata_temp.obs.columns:
    sc.pl.umap(
        adata_temp,
        color='cell_type',
        show=False,
        title='UMAP - Unintegrated (by cell type)',
        save='_unintegrated_celltype.png'
    )
    print(f"   Saved: {fig_dir}/umap_unintegrated_celltype.png")

# Clean up temporary object
del adata_temp
print("\nUnintegrated baseline analysis completed")


Generating unintegrated visualizations...
   Saved: /home/h2048/data/py/1125/bbknn_output_optimized/figures/umap_unintegrated_batch.png
   Saved: /home/h2048/data/py/1125/bbknn_output_optimized/figures/umap_unintegrated_celltype.png

Unintegrated baseline analysis completed


## 7. PCA on Preprocessed Data

In [55]:
print("\n" + "="*70)
print("Step 4: Running PCA")
print("="*70)

print(f"\nRunning PCA (n_comps={N_PCS})...")
sc.tl.pca(adata_hvg, n_comps=N_PCS, svd_solver='arpack')

# Calculate explained variance ratio
var_ratio = adata_hvg.uns['pca']['variance_ratio']
cumsum_var = np.cumsum(var_ratio)

print(f"   PC1-10 explained variance: {cumsum_var[9]:.2%}")
print(f"   PC1-20 explained variance: {cumsum_var[19]:.2%}")
print(f"   PC1-50 explained variance: {cumsum_var[49]:.2%}")
print("\nPCA completed")


Step 4: Running PCA

Running PCA (n_comps=50)...
   PC1-10 explained variance: 22.64%
   PC1-20 explained variance: 28.84%
   PC1-50 explained variance: 35.70%

PCA completed


## 8. BBKNN Batch Integration

In [56]:
print("\n" + "="*70)
print("Step 5: BBKNN Batch Integration")
print("="*70)

print("\nBBKNN parameters:")
print(f"   batch_key: {BATCH_KEY}")
print(f"   neighbors_within_batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")
print(f"   n_pcs: {BBKNN_N_PCS}")

print("\nRunning BBKNN...")
print("   BBKNN constructs batch-balanced k-nearest neighbor graph")
print("   This may take a few minutes depending on data size...")

bbknn_start = time.time()

# Run BBKNN
# BBKNN will directly modify adata neighbors information
bbknn.bbknn(
    adata_hvg,
    batch_key=BATCH_KEY,
    neighbors_within_batch=BBKNN_NEIGHBORS_WITHIN_BATCH,
    n_pcs=BBKNN_N_PCS,
    copy=False  # Modify adata in place
)

bbknn_time = time.time() - bbknn_start
print(f"   BBKNN completed in {bbknn_time:.1f} seconds")

print("\n   Batch-balanced neighbor graph constructed")
print("   Ready for downstream analysis (UMAP, clustering)")


Step 5: BBKNN Batch Integration

BBKNN parameters:
   batch_key: dataset
   neighbors_within_batch: 3
   n_pcs: 50

Running BBKNN...
   BBKNN constructs batch-balanced k-nearest neighbor graph
   This may take a few minutes depending on data size...
   BBKNN completed in 726.0 seconds

   Batch-balanced neighbor graph constructed
   Ready for downstream analysis (UMAP, clustering)


## 9. UMAP Dimensionality Reduction

In [57]:
print("\n" + "="*70)
print("Step 6: UMAP and Clustering")
print("="*70)

if RUN_UMAP:
    print("\nComputing UMAP...")
    print(f"   min_dist: {UMAP_MIN_DIST}")
    print("   Using BBKNN neighbor graph")
    
    # BBKNN has already computed neighbors, directly use UMAP
    sc.tl.umap(adata_hvg, min_dist=UMAP_MIN_DIST)
    print("   UMAP completed")
else:
    print("\nSkipping UMAP (RUN_UMAP=False)")


Step 6: UMAP and Clustering

Computing UMAP...
   min_dist: 0.5
   Using BBKNN neighbor graph
   UMAP completed


## 10. Multi-resolution Clustering

In [ ]:
if RUN_CLUSTERING:
    print("\nRunning multi-resolution Leiden clustering...")
    print(f"   Resolutions: {LEIDEN_RESOLUTIONS}")
    
    for res in LEIDEN_RESOLUTIONS:
        key = f'leiden_bbknn_res{res}'
        sc.tl.leiden(adata_hvg, resolution=res, key_added=key)
        n_clusters = adata_hvg.obs[key].nunique()
        print(f"   Resolution {res}: {n_clusters} clusters")
    
    # Set default clustering
    default_key = f'leiden_bbknn_res{DEFAULT_RESOLUTION}'
    if default_key in adata_hvg.obs.columns:
        adata_hvg.obs['leiden_bbknn'] = adata_hvg.obs[default_key]
        print(f"\n   Default clustering: {default_key}")
else:
    print("\nSkipping clustering (RUN_CLUSTERING=False)")


Running multi-resolution Leiden clustering...
   Resolutions: [1.0, 1.4, 1.8, 2.2]


## 11. Basic Visualizations

In [ ]:
print("\n" + "="*70)
print("Step 7: Generating Visualizations")
print("="*70)

if not RUN_UMAP:
    print("Skipping visualizations (UMAP not computed)")
else:
    # Basic UMAP plots
    print("\nGenerating UMAP plots...")
    for var in VISUALIZATION_VARS:
        if var in adata_hvg.obs.columns:
            print(f"   - UMAP colored by {var}")
            sc.pl.umap(
                adata_hvg,
                color=var,
                show=False,
                title=f'UMAP - {var}',
                save=f'_{var}.png'
            )
    
    print(f"\nFigures saved to: {fig_dir}")

In [ ]:
# Clustering results visualization
if RUN_CLUSTERING and RUN_UMAP:
    print("\nGenerating clustering plots...")
    for res in LEIDEN_RESOLUTIONS:
        key = f'leiden_bbknn_res{res}'
        if key in adata_hvg.obs.columns:
            print(f"   - UMAP colored by {key}")
            sc.pl.umap(
                adata_hvg,
                color=key,
                show=False,
                title=f'UMAP - Leiden (res={res})',
                save=f'_{key}.png'
            )

## 12. Facet Plots by Batch

In [ ]:
# Facet plots by batch
if GENERATE_FACET_PLOTS and BATCH_KEY in adata_hvg.obs.columns and RUN_UMAP:
    print("\nGenerating facet plots...")
    batches = adata_hvg.obs[BATCH_KEY].unique()
    
    if len(batches) <= 10:
        try:
            print(f"   - Facet plot by {BATCH_KEY}")
            
            n_batches = len(batches)
            n_cols = min(3, n_batches)
            n_rows = (n_batches + n_cols - 1) // n_cols
            
            fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 5*n_rows))
            if n_batches == 1:
                axes = [axes]
            else:
                axes = axes.flatten()
            
            color_var = 'leiden_bbknn' if 'leiden_bbknn' in adata_hvg.obs.columns else None
            
            for i, batch in enumerate(sorted(batches)):
                adata_batch = adata_hvg[adata_hvg.obs[BATCH_KEY] == batch]
                sc.pl.umap(
                    adata_batch,
                    color=color_var,
                    ax=axes[i],
                    show=False,
                    title=f'{batch}'
                )
            
            # Hide extra subplots
            for i in range(n_batches, len(axes)):
                axes[i].axis('off')
            
            plt.tight_layout()
            plt.savefig(fig_dir / f'umap_facet_by_{BATCH_KEY}.png', dpi=150, bbox_inches='tight')
            plt.close()
            
            print(f"   Saved: {fig_dir}/umap_facet_by_{BATCH_KEY}.png")
            
        except Exception as e:
            print(f"   Warning: Facet plot failed - {e}")
    else:
        print(f"   Skipping facet plots (too many batches: {len(batches)})")

## 13. Evaluate Integration Quality (Optional)

In [ ]:
print("\n" + "="*70)
print("Step 8: Evaluating Integration Quality")
print("="*70)

eval_results = {}

if EVALUATE_INTEGRATION:
    # Silhouette score (batch) - using PCA space
    print("\nComputing silhouette scores...")
    
    if BATCH_KEY in adata_hvg.obs.columns and 'X_pca' in adata_hvg.obsm:
        batch_labels = adata_hvg.obs[BATCH_KEY].astype('category').cat.codes
        
        # Evaluate using PCA space
        sil_batch = silhouette_score(adata_hvg.obsm['X_pca'][:, :BBKNN_N_PCS], batch_labels)
        eval_results['silhouette_batch'] = float(sil_batch)
        print(f"   Silhouette (batch): {sil_batch:.4f}")
        print(f"   Interpretation: closer to 0 = better batch mixing")
    
    # If cell type information is available, compute biological conservation
    if 'cell_type' in adata_hvg.obs.columns and 'X_pca' in adata_hvg.obsm:
        print("\nComputing biological conservation...")
        cell_type_labels = adata_hvg.obs['cell_type'].astype('category').cat.codes
        
        sil_bio = silhouette_score(adata_hvg.obsm['X_pca'][:, :BBKNN_N_PCS], cell_type_labels)
        eval_results['silhouette_biology'] = float(sil_bio)
        print(f"   Silhouette (cell type): {sil_bio:.4f}")
        print(f"   Interpretation: closer to 1 = better biological separation")
    
    # Compute batch mixing entropy
    if 'leiden_bbknn' in adata_hvg.obs.columns:
        print("\nComputing batch mixing metrics...")
        
        # For each cluster, compute entropy of batch distribution
        clusters = adata_hvg.obs['leiden_bbknn'].unique()
        entropies = []
        
        for cluster in clusters:
            cluster_mask = adata_hvg.obs['leiden_bbknn'] == cluster
            batch_dist = adata_hvg.obs.loc[cluster_mask, BATCH_KEY].value_counts(normalize=True)
            ent = entropy(batch_dist)
            entropies.append(ent)
        
        mean_entropy = np.mean(entropies)
        eval_results['mean_cluster_batch_entropy'] = float(mean_entropy)
        print(f"   Mean cluster batch entropy: {mean_entropy:.4f}")
        print(f"   Interpretation: higher = better batch mixing within clusters")
    
    # Save results
    eval_path = output_dir / "integration_evaluation.json"
    with open(eval_path, 'w') as f:
        json.dump(eval_results, f, indent=2)
    
    print(f"\nEvaluation results saved to: {eval_path}")
else:
    print("\nSkipping integration evaluation (EVALUATE_INTEGRATION=False)")

## 14. Transfer Results to Full Dataset

In [ ]:
print("\n" + "="*70)
print("Step 9: Transferring Results to Full Dataset")
print("="*70)

print("\nTransferring results to full dataset...")

# Transfer embeddings
for key in ['X_pca', 'X_umap']:
    if key in adata_hvg.obsm:
        adata.obsm[key] = adata_hvg.obsm[key]
        print(f"   Transferred {key}")

# Transfer clustering results
for col in adata_hvg.obs.columns:
    if col.startswith('leiden_bbknn'):
        adata.obs[col] = adata_hvg.obs[col]
        print(f"   Transferred {col}")

# Copy neighbors information
if 'neighbors' in adata_hvg.uns:
    adata.uns['neighbors'] = adata_hvg.uns['neighbors']
    print("   Transferred neighbors information")

if 'connectivities' in adata_hvg.obsp:
    adata.obsp['connectivities'] = adata_hvg.obsp['connectivities']
    print("   Transferred connectivities")

if 'distances' in adata_hvg.obsp:
    adata.obsp['distances'] = adata_hvg.obsp['distances']
    print("   Transferred distances")

print("\nTransfer completed")

## 15. Save Integrated Data

In [ ]:
print("\n" + "="*70)
print("Step 10: Saving Integrated Data")
print("="*70)

print("\nSaving integrated data...")
final_path = output_dir / "adata_bbknn_integrated.h5ad"

print("   Using gzip compression (level 9)...")
adata.write_h5ad(final_path, compression='gzip', compression_opts=9)

file_size = final_path.stat().st_size / (1024**3)
print(f"   Saved: {final_path} ({file_size:.2f} GB)")

## 16. Generate Summary Report

In [ ]:
print("\n" + "="*70)
print("Step 11: Generating Summary Report")
print("="*70)

# Calculate total processing time
total_time = time.time() - start_time

# Generate report
report = []
report.append("="*70)
report.append("BBKNN Batch Integration - Analysis Summary")
report.append("="*70)
report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report.append(f"Total processing time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")
report.append("")

# Data overview
report.append("[ Data Overview ]")
report.append(f"  Cells: {adata.n_obs:,}")
report.append(f"  Genes: {adata.n_vars:,}")
report.append("")

# Batch information
if BATCH_KEY in adata.obs.columns:
    report.append(f"[ Batch Distribution (key: {BATCH_KEY}) ]")
    batch_counts = adata.obs[BATCH_KEY].value_counts().sort_index()
    for batch, count in batch_counts.items():
        pct = count / adata.n_obs * 100
        report.append(f"  {batch}: {count:,} cells ({pct:.1f}%)")
    report.append(f"  Total batches: {len(batch_counts)}")
    report.append("")

# Integration quality metrics
if eval_results:
    report.append("[ Integration Quality Metrics ]")
    if 'silhouette_batch' in eval_results:
        report.append(f"  Silhouette Score (batch): {eval_results['silhouette_batch']:.4f}")
        report.append("  (Interpretation: closer to 0 = better batch mixing)")
    if 'silhouette_biology' in eval_results:
        report.append(f"  Silhouette Score (cell type): {eval_results['silhouette_biology']:.4f}")
        report.append("  (Interpretation: closer to 1 = better biological separation)")
    if 'mean_cluster_batch_entropy' in eval_results:
        report.append(f"  Mean cluster batch entropy: {eval_results['mean_cluster_batch_entropy']:.4f}")
        report.append("  (Interpretation: higher = better batch mixing)")
    report.append("")

# Multi-resolution clustering
if RUN_CLUSTERING:
    report.append("[ Multi-resolution Clustering Results ]")
    for res in LEIDEN_RESOLUTIONS:
        key = f'leiden_bbknn_res{res}'
        if key in adata.obs.columns:
            n_clusters = adata.obs[key].nunique()
            default_marker = " (default)" if res == DEFAULT_RESOLUTION else ""
            report.append(f"  Resolution {res}: {n_clusters} clusters{default_marker}")
    report.append("")

# Available embeddings
report.append("[ Available Embeddings ]")
for key in adata.obsm.keys():
    report.append(f"  - {key}: {adata.obsm[key].shape}")
report.append("")

# BBKNN configuration
report.append("[ BBKNN Configuration ]")
report.append(f"  neighbors_within_batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")
report.append(f"  n_pcs: {BBKNN_N_PCS}")
report.append(f"  batch_key: {BATCH_KEY}")
report.append(f"  High variable genes: {USE_HVG} (n={N_TOP_GENES if USE_HVG else 'N/A'})")
report.append("")

# Output files
report.append("[ Output Files ]")
report.append(f"  - Data: {output_dir / 'adata_bbknn_integrated.h5ad'}")
report.append(f"  - Figures: {output_dir / 'figures/'}*.png")
if EVALUATE_INTEGRATION:
    report.append(f"  - Evaluation: {output_dir / 'integration_evaluation.json'}")
report.append("")

# Method notes
report.append("[ Method Notes ]")
report.append("  BBKNN (Batch Balanced k-Nearest Neighbors):")
report.append("  - Graph-based batch correction method")
report.append("  - Constructs batch-balanced neighbor graph")
report.append("  - Fast, no GPU required")
report.append("  - Suitable for well-separated batches")
report.append("")

report.append("="*70)
report.append("Analysis completed successfully")
report.append("="*70)

report_text = '\n'.join(report)

# Save report
report_path = output_dir / "analysis_summary.txt"
with open(report_path, 'w') as f:
    f.write(report_text)

print(f"\nReport saved to: {report_path}")
print("\n" + report_text)

## 17. Final Summary

In [ ]:
print("\n" + "="*70)
print("All analyses completed successfully")
print("="*70)
print(f"\nOutput directory: {output_dir}")
print(f"Data: {final_path} ({file_size:.2f} GB)")
if RUN_UMAP:
    print(f"Figures: {output_dir / 'figures/'}*.png")
if EVALUATE_INTEGRATION:
    print(f"Evaluation: {output_dir / 'integration_evaluation.json'}")
print(f"\nTotal time: {total_time:.1f} seconds ({total_time/60:.1f} minutes)")

print(f"\nKey parameters:")
print(f"   BBKNN neighbors_within_batch: {BBKNN_NEIGHBORS_WITHIN_BATCH}")
print(f"   High variable genes: {N_TOP_GENES if USE_HVG else 'Not used'}")
print(f"   Multi-resolution clustering: {LEIDEN_RESOLUTIONS}")
print()

---

## 18. Annotation Level Analysis - Self-Match Harmonization

**Strategy:** For each annotation level, perform self-match harmonization within each dataset, then combine results

### 18.1 Check Available Annotation Levels

In [ ]:
print("\n" + "="*70)
print("Step 12: Annotation Level Analysis")
print("="*70)

# Define annotation levels to analyze
ANNOTATION_LEVELS = [
    'ann_finest_level',
    'ann_level_1',
    'ann_level_2',
    'ann_level_3',
    'ann_level_4',
    'ann_level_5'
]

# Check which annotation levels exist in the data
available_ann_levels = []
print("\nChecking available annotation levels:")
for ann_level in ANNOTATION_LEVELS:
    if ann_level in adata.obs.columns:
        n_types = adata.obs[ann_level].nunique()
        print(f"   ✓ {ann_level}: {n_types} unique cell types")
        available_ann_levels.append(ann_level)
    else:
        print(f"   ✗ {ann_level}: not found")

if not available_ann_levels:
    print("\nNo annotation levels found in data. Skipping this analysis.")
else:
    print(f"\nFound {len(available_ann_levels)} annotation levels for analysis")

### 18.2 Examine Dataset x Annotation Distribution

In [ ]:
if available_ann_levels:
    print("\nDataset x Annotation Level Distribution:")
    print("="*70)
    
    for ann_level in available_ann_levels:
        print(f"\n{ann_level}:")
        print("-" * 50)
        
        # Create cross-tabulation
        crosstab = pd.crosstab(
            adata.obs[BATCH_KEY],
            adata.obs[ann_level],
            margins=True
        )
        
        print(crosstab)
        print()

### 18.3 Self-Match Analysis Configuration

In [ ]:
if available_ann_levels:
    # Configuration for self-match analysis
    SELFMATCH_CONFIG = {
        'run_selfmatch': True,  # Whether to run self-match analysis
        'min_cells_per_type': 50,  # Minimum cells per cell type in each dataset
        'bbknn_neighbors': 3,  # BBKNN neighbors for self-match (lower than global)
        'umap_min_dist': 0.3,  # UMAP min_dist for self-match
        'clustering_resolution': 0.4,  # Leiden resolution for self-match
    }
    
    print("Self-Match Analysis Configuration:")
    for key, value in SELFMATCH_CONFIG.items():
        print(f"   {key}: {value}")
    
    # Create output directory for self-match results
    selfmatch_dir = output_dir / "selfmatch_analysis"
    selfmatch_dir.mkdir(exist_ok=True)
    print(f"\nSelf-match results will be saved to: {selfmatch_dir}")

### 18.4 Perform Self-Match Harmonization for Each Annotation Level

In [ ]:
if available_ann_levels and SELFMATCH_CONFIG['run_selfmatch']:
    print("\n" + "="*70)
    print("Running Self-Match Harmonization")
    print("="*70)
    
    # Dictionary to store results
    selfmatch_results = {}
    
    for ann_level in available_ann_levels:
        print(f"\n{'='*70}")
        print(f"Processing: {ann_level}")
        print(f"{'='*70}")
        
        # Get unique cell types in this annotation level
        cell_types = adata.obs[ann_level].unique()
        print(f"\nTotal cell types: {len(cell_types)}")
        
        # Initialize results for this annotation level
        selfmatch_results[ann_level] = {}
        
        # Process each cell type
        for cell_type in cell_types:
            print(f"\n{'-'*60}")
            print(f"Cell type: {cell_type}")
            print(f"{'-'*60}")
            
            # Subset data for this cell type
            mask = adata.obs[ann_level] == cell_type
            adata_subset = adata[mask].copy()
            
            # Check cell counts per dataset
            dataset_counts = adata_subset.obs[BATCH_KEY].value_counts()
            print(f"\nCells per dataset:")
            for dataset, count in dataset_counts.items():
                print(f"   {dataset}: {count:,} cells")
            
            # Check if we have enough cells and multiple datasets
            if len(dataset_counts) < 2:
                print(f"   ⚠ Skipping: only 1 dataset present")
                continue
            
            if adata_subset.n_obs < SELFMATCH_CONFIG['min_cells_per_type']:
                print(f"   ⚠ Skipping: too few cells ({adata_subset.n_obs} < {SELFMATCH_CONFIG['min_cells_per_type']})")
                continue
            
            # Check if any dataset has too few cells
            min_dataset_cells = dataset_counts.min()
            if min_dataset_cells < 3:  # BBKNN minimum requirement
                print(f"   ⚠ Skipping: dataset with too few cells ({min_dataset_cells} < 3)")
                continue
            
            print(f"\n   ✓ Processing {adata_subset.n_obs:,} cells across {len(dataset_counts)} datasets")
            
            try:
                # Preprocessing for this subset
                print("   - Preprocessing...")
                
                # Use raw counts if available
                if 'counts' in adata_subset.layers:
                    adata_subset.X = adata_subset.layers['counts'].copy()
                
                # Basic filtering
                sc.pp.filter_genes(adata_subset, min_cells=3)
                
                # Normalize and log transform
                sc.pp.normalize_total(adata_subset, target_sum=1e4)
                sc.pp.log1p(adata_subset)
                
                # Select HVGs
                sc.pp.highly_variable_genes(
                    adata_subset,
                    n_top_genes=min(2000, adata_subset.n_vars),
                    flavor='seurat_v3',
                    batch_key=BATCH_KEY,
                    subset=True
                )
                
                # Scale
                sc.pp.scale(adata_subset, max_value=10)
                
                # PCA
                print("   - Running PCA...")
                n_pcs = min(30, adata_subset.n_obs - 1, adata_subset.n_vars)
                sc.tl.pca(adata_subset, n_comps=n_pcs)
                
                # BBKNN integration
                print("   - Running BBKNN...")
                bbknn.bbknn(
                    adata_subset,
                    batch_key=BATCH_KEY,
                    neighbors_within_batch=SELFMATCH_CONFIG['bbknn_neighbors'],
                    n_pcs=n_pcs,
                    copy=False
                )
                
                # UMAP
                print("   - Computing UMAP...")
                sc.tl.umap(adata_subset, min_dist=SELFMATCH_CONFIG['umap_min_dist'])
                
                # Clustering
                print("   - Clustering...")
                sc.tl.leiden(
                    adata_subset,
                    resolution=SELFMATCH_CONFIG['clustering_resolution'],
                    key_added='leiden_selfmatch'
                )
                
                n_clusters = adata_subset.obs['leiden_selfmatch'].nunique()
                print(f"   ✓ Completed: {n_clusters} sub-clusters identified")
                
                # Store results
                selfmatch_results[ann_level][cell_type] = {
                    'adata': adata_subset,
                    'n_cells': adata_subset.n_obs,
                    'n_datasets': len(dataset_counts),
                    'n_clusters': n_clusters,
                    'dataset_counts': dataset_counts.to_dict()
                }
                
            except Exception as e:
                print(f"   ✗ Error: {str(e)}")
                continue
        
        print(f"\nCompleted {ann_level}: {len(selfmatch_results[ann_level])} cell types processed")
    
    print("\n" + "="*70)
    print("Self-Match Harmonization Completed")
    print("="*70)

### 18.5 Visualize Self-Match Results

In [ ]:
if available_ann_levels and SELFMATCH_CONFIG['run_selfmatch']:
    print("\n" + "="*70)
    print("Generating Self-Match Visualizations")
    print("="*70)
    
    for ann_level, cell_type_results in selfmatch_results.items():
        if not cell_type_results:
            print(f"\nNo results for {ann_level}, skipping...")
            continue
        
        print(f"\n{'='*70}")
        print(f"Visualizing: {ann_level}")
        print(f"{'='*70}")
        
        # Create annotation-specific directory
        ann_fig_dir = selfmatch_dir / ann_level
        ann_fig_dir.mkdir(exist_ok=True)
        
        for cell_type, result_dict in cell_type_results.items():
            adata_subset = result_dict['adata']
            
            # Safe filename
            safe_celltype = cell_type.replace('/', '_').replace(' ', '_')
            
            print(f"\n   Plotting: {cell_type}")
            
            # Create figure with multiple panels
            fig, axes = plt.subplots(1, 3, figsize=(18, 5))
            
            # Panel 1: Colored by dataset
            sc.pl.umap(
                adata_subset,
                color=BATCH_KEY,
                ax=axes[0],
                show=False,
                title=f'{cell_type}\n(by dataset)'
            )
            
            # Panel 2: Colored by sub-clusters
            sc.pl.umap(
                adata_subset,
                color='leiden_selfmatch',
                ax=axes[1],
                show=False,
                title=f'{cell_type}\n(sub-clusters)'
            )
            
            # Panel 3: Cell counts bar plot
            dataset_counts = pd.Series(result_dict['dataset_counts'])
            dataset_counts.plot(kind='bar', ax=axes[2])
            axes[2].set_title(f'{cell_type}\n(cell counts)')
            axes[2].set_xlabel('Dataset')
            axes[2].set_ylabel('Number of cells')
            axes[2].tick_params(axis='x', rotation=45)
            
            plt.tight_layout()
            
            # Save figure
            fig_path = ann_fig_dir / f'{safe_celltype}_selfmatch.png'
            plt.savefig(fig_path, dpi=150, bbox_inches='tight')
            plt.close()
            
            print(f"      Saved: {fig_path.name}")
        
        print(f"\n   Figures saved to: {ann_fig_dir}")
    
    print("\n" + "="*70)
    print("Visualization Completed")
    print("="*70)

### 18.6 Generate Self-Match Summary Report

In [ ]:
if available_ann_levels and SELFMATCH_CONFIG['run_selfmatch']:
    print("\n" + "="*70)
    print("Generating Self-Match Summary Report")
    print("="*70)
    
    summary_report = []
    summary_report.append("="*70)
    summary_report.append("Self-Match Harmonization Summary Report")
    summary_report.append("="*70)
    summary_report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    summary_report.append("")
    summary_report.append(f"Batch key: {BATCH_KEY}")
    summary_report.append(f"Total datasets: {adata.obs[BATCH_KEY].nunique()}")
    summary_report.append("")
    
    # Configuration
    summary_report.append("[ Configuration ]")
    for key, value in SELFMATCH_CONFIG.items():
        summary_report.append(f"  {key}: {value}")
    summary_report.append("")
    
    # Results by annotation level
    for ann_level in available_ann_levels:
        summary_report.append(f"[ {ann_level} ]")
        
        if ann_level not in selfmatch_results or not selfmatch_results[ann_level]:
            summary_report.append("  No results (all cell types skipped)")
            summary_report.append("")
            continue
        
        cell_type_results = selfmatch_results[ann_level]
        total_cell_types = adata.obs[ann_level].nunique()
        processed_cell_types = len(cell_type_results)
        
        summary_report.append(f"  Total cell types: {total_cell_types}")
        summary_report.append(f"  Processed cell types: {processed_cell_types}")
        summary_report.append(f"  Skipped cell types: {total_cell_types - processed_cell_types}")
        summary_report.append("")
        
        # Details for each cell type
        summary_report.append("  Cell type details:")
        for cell_type, result_dict in sorted(cell_type_results.items()):
            summary_report.append(f"    - {cell_type}:")
            summary_report.append(f"        Cells: {result_dict['n_cells']:,}")
            summary_report.append(f"        Datasets: {result_dict['n_datasets']}")
            summary_report.append(f"        Sub-clusters: {result_dict['n_clusters']}")
            
            # Dataset distribution
            dataset_dist = result_dict['dataset_counts']
            for dataset, count in sorted(dataset_dist.items()):
                summary_report.append(f"          {dataset}: {count:,} cells")
        
        summary_report.append("")
    
    # Summary statistics
    summary_report.append("[ Overall Statistics ]")
    total_processed = sum(len(results) for results in selfmatch_results.values())
    total_cells_analyzed = sum(
        result['n_cells']
        for ann_results in selfmatch_results.values()
        for result in ann_results.values()
    )
    summary_report.append(f"  Total cell types processed: {total_processed}")
    summary_report.append(f"  Total cells analyzed: {total_cells_analyzed:,}")
    summary_report.append("")
    
    summary_report.append("="*70)
    summary_report.append("End of Report")
    summary_report.append("="*70)
    
    # Save report
    summary_text = '\n'.join(summary_report)
    summary_path = selfmatch_dir / "selfmatch_summary.txt"
    with open(summary_path, 'w') as f:
        f.write(summary_text)
    
    print(f"\nReport saved to: {summary_path}")
    print("\n" + summary_text)

### 18.7 Save Self-Match Results

In [ ]:
if available_ann_levels and SELFMATCH_CONFIG['run_selfmatch']:
    print("\n" + "="*70)
    print("Saving Self-Match Results")
    print("="*70)
    
    # Save each cell type's harmonized data
    for ann_level, cell_type_results in selfmatch_results.items():
        if not cell_type_results:
            continue
        
        print(f"\nSaving results for {ann_level}...")
        
        # Create annotation-specific directory
        ann_data_dir = selfmatch_dir / ann_level / "data"
        ann_data_dir.mkdir(parents=True, exist_ok=True)
        
        for cell_type, result_dict in cell_type_results.items():
            adata_subset = result_dict['adata']
            
            # Safe filename
            safe_celltype = cell_type.replace('/', '_').replace(' ', '_')
            
            # Save as h5ad
            save_path = ann_data_dir / f'{safe_celltype}_selfmatch.h5ad'
            adata_subset.write_h5ad(save_path, compression='gzip')
            
            file_size = save_path.stat().st_size / (1024**2)  # MB
            print(f"   Saved: {save_path.name} ({file_size:.1f} MB)")
        
        print(f"   Data saved to: {ann_data_dir}")
    
    print("\n" + "="*70)
    print("Self-Match Results Saved")
    print("="*70)

### 18.8 Transfer Self-Match Sub-cluster Labels to Main Object

In [ ]:
if available_ann_levels and SELFMATCH_CONFIG['run_selfmatch']:
    print("\n" + "="*70)
    print("Transferring Sub-cluster Labels to Main Object")
    print("="*70)
    
    # For each annotation level, create a new column with sub-cluster information
    for ann_level, cell_type_results in selfmatch_results.items():
        if not cell_type_results:
            continue
        
        print(f"\nProcessing {ann_level}...")
        
        # Create new column for sub-clusters
        subcluster_col = f'{ann_level}_subcluster'
        adata.obs[subcluster_col] = adata.obs[ann_level].astype(str)
        
        # Transfer sub-cluster labels
        for cell_type, result_dict in cell_type_results.items():
            adata_subset = result_dict['adata']
            
            # Find cells of this type in main object
            cell_mask = adata.obs[ann_level] == cell_type
            
            # Create sub-cluster labels: celltype_cluster
            subcluster_labels = adata_subset.obs['leiden_selfmatch'].astype(str)
            subcluster_labels = cell_type + '_' + subcluster_labels
            
            # Assign to main object
            adata.obs.loc[cell_mask, subcluster_col] = subcluster_labels.values
            
            n_subclusters = adata_subset.obs['leiden_selfmatch'].nunique()
            print(f"   {cell_type}: {n_subclusters} sub-clusters assigned")
        
        # Summary
        n_unique = adata.obs[subcluster_col].nunique()
        print(f"\n   Total unique sub-clusters in {subcluster_col}: {n_unique}")
    
    print("\n" + "="*70)
    print("Sub-cluster Labels Transferred")
    print("="*70)

### 18.9 Visualize Sub-cluster Results on Main UMAP

In [ ]:
if available_ann_levels and SELFMATCH_CONFIG['run_selfmatch'] and 'X_umap' in adata.obsm:
    print("\n" + "="*70)
    print("Visualizing Sub-clusters on Main UMAP")
    print("="*70)
    
    subcluster_fig_dir = selfmatch_dir / "main_umap_subclusters"
    subcluster_fig_dir.mkdir(exist_ok=True)
    
    for ann_level in available_ann_levels:
        subcluster_col = f'{ann_level}_subcluster'
        
        if subcluster_col not in adata.obs.columns:
            continue
        
        print(f"\nPlotting {subcluster_col}...")
        
        n_unique = adata.obs[subcluster_col].nunique()
        
        # Create figure
        fig, axes = plt.subplots(1, 2, figsize=(16, 6))
        
        # Original annotation
        sc.pl.umap(
            adata,
            color=ann_level,
            ax=axes[0],
            show=False,
            title=f'{ann_level}\n(original, n={adata.obs[ann_level].nunique()})'
        )
        
        # Sub-clusters
        sc.pl.umap(
            adata,
            color=subcluster_col,
            ax=axes[1],
            show=False,
            title=f'{subcluster_col}\n(harmonized, n={n_unique})',
            legend_loc='right margin',
            legend_fontsize=6
        )
        
        plt.tight_layout()
        
        # Save
        fig_path = subcluster_fig_dir / f'{ann_level}_subclusters_comparison.png'
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"   Saved: {fig_path.name}")
    
    print(f"\n   Figures saved to: {subcluster_fig_dir}")
    print("\n" + "="*70)
    print("Visualization Completed")
    print("="*70)

### 18.10 Save Final Data with Sub-clusters

In [ ]:
if available_ann_levels and SELFMATCH_CONFIG['run_selfmatch']:
    print("\n" + "="*70)
    print("Saving Final Data with Sub-cluster Annotations")
    print("="*70)
    
    final_path_with_subclusters = output_dir / "adata_bbknn_integrated_with_subclusters.h5ad"
    
    print("\nSaving data with sub-cluster annotations...")
    print("   Using gzip compression (level 9)...")
    adata.write_h5ad(final_path_with_subclusters, compression='gzip', compression_opts=9)
    
    file_size = final_path_with_subclusters.stat().st_size / (1024**3)
    print(f"   Saved: {final_path_with_subclusters} ({file_size:.2f} GB)")
    
    # List new columns added
    print("\nNew annotation columns added:")
    for ann_level in available_ann_levels:
        subcluster_col = f'{ann_level}_subcluster'
        if subcluster_col in adata.obs.columns:
            n_unique = adata.obs[subcluster_col].nunique()
            print(f"   - {subcluster_col}: {n_unique} unique sub-clusters")
    
    print("\n" + "="*70)
    print("Final Data Saved")
    print("="*70)

# Cell 1: Import CellHint and Check Installation

In [ ]:
try:
    import cellhint
    print(f"CellHint version: {cellhint.__version__}")
    print("CellHint loaded successfully")
except ImportError:
    print("ERROR: CellHint not installed")
    print("Install with: pip install cellhint")
    raise

import matplotlib.pyplot as plt
import seaborn as sns

Cell 2: Select Annotation Level for Harmonization

In [ ]:
# Define annotation level priority
ANNOTATION_LEVELS = [
    'ann_finest_level',
    'ann_level_5',
    'ann_level_4', 
    'ann_level_3',
    'ann_level_2',
    'ann_level_1'
]

# Select annotation level with priority
selected_ann_level = None
for ann_level in ANNOTATION_LEVELS:
    if ann_level in adata.obs.columns:
        selected_ann_level = ann_level
        print(f"✓ Selected annotation level: {selected_ann_level}")
        break

if selected_ann_level is None:
    raise ValueError("No annotation levels found in adata.obs")

# Display annotation statistics
print(f"\nAnnotation Statistics for '{selected_ann_level}':")
print(f"  Total cell types: {adata.obs[selected_ann_level].nunique()}")
print(f"  Total cells: {adata.n_obs:,}")

# Show cell type distribution
cell_type_counts = adata.obs[selected_ann_level].value_counts()
print(f"\nTop 10 cell types:")
for idx, (ct, count) in enumerate(cell_type_counts.head(10).items(), 1):
    pct = count / adata.n_obs * 100
    print(f"  {idx}. {ct}: {count:,} cells ({pct:.2f}%)")

# Show batch distribution
batch_counts = adata.obs[BATCH_KEY].value_counts()
print(f"\nBatch distribution (key: {BATCH_KEY}):")
for batch, count in batch_counts.items():
    pct = count / adata.n_obs * 100
    print(f"  {batch}: {count:,} cells ({pct:.2f}%)")

Cell 3: Prepare Data for CellHint Harmonization

In [ ]:

print("Checking data requirements for CellHint...")

# Check if we have the required data
checks = {
    'X data': adata.X is not None,
    'Batch key exists': BATCH_KEY in adata.obs.columns,
    'Cell type key exists': selected_ann_level in adata.obs.columns,
    'PCA exists': 'X_pca' in adata.obsm,
}

print("\nData checks:")
for check, status in checks.items():
    symbol = "✓" if status else "✗"
    print(f"  {symbol} {check}")

if not all(checks.values()):
    print("\nWARNING: Some required data is missing")
else:
    print("\n✓ All requirements met")

# Create a working copy for harmonization
adata_harm = adata.copy()

print(f"\nWorking copy created:")
print(f"  Cells: {adata_harm.n_obs:,}")
print(f"  Genes: {adata_harm.n_vars:,}")

Cell 4: Run CellHint Harmonization

In [ ]:
print("="*70)
print("Running CellHint Harmonization")
print("="*70)

# CellHint harmonization parameters
CELLHINT_PARAMS = {
    'use_rep': 'X_pca',  # Use PCA representation
    'metric': 'euclidean',  # Distance metric
    'use_pct': False,  # Set to True for large batch effects
    'filter_cells': False,  # Keep all cells
    'normalize': True,  # Normalize distance matrix
    'Gaussian_kernel': False,  # Apply Gaussian kernel
    'F_test_prune': True,  # Prune PCT tree with F-test
    'p_thres': 0.05,  # p-value threshold for pruning
    'reorder_dataset': True,  # Reorder datasets by similarity
    'minimum_unique_percents': (0.4, 0.5, 0.6, 0.7, 0.8),  # Try multiple thresholds
    'minimum_divide_percents': (0.1, 0.15, 0.2),  # Try multiple thresholds
    'maximum_novel_percent': 0.05,  # Threshold for novel cell types
    'reannotate': True,  # Reannotate cells with harmonized labels
    'prefix': 'harm_',  # Prefix for harmonized annotation columns
    'random_state': 42,  # For reproducibility
}

print("Harmonization parameters:")
for key, value in CELLHINT_PARAMS.items():
    print(f"  {key}: {value}")

# Run harmonization
print("\nRunning harmonization... (this may take several minutes)")
import time
start_time = time.time()

alignment = cellhint.harmonize(
    adata_harm,
    dataset=BATCH_KEY,
    cell_type=selected_ann_level,
    **CELLHINT_PARAMS
)

elapsed_time = time.time() - start_time
print(f"\n✓ Harmonization completed in {elapsed_time:.1f} seconds ({elapsed_time/60:.1f} minutes)")

# Display harmonization results
print("\n" + "="*70)
print("Harmonization Results")
print("="*70)

# Show relation table
print("\nHarmonization relation table:")
print(alignment.relation)

# Show groups
if hasattr(alignment, 'groups') and alignment.groups is not None:
    print("\nCell type groups:")
    print(alignment.groups)

# Show reannotation results
if hasattr(alignment, 'reannotation') and alignment.reannotation is not None:
    print("\nReannotation summary:")
    print(f"  Total cells reannotated: {len(alignment.reannotation)}")
    print(f"  New columns: {list(alignment.reannotation.columns)}")
    
    # Transfer harmonized annotations back to original adata
    for col in alignment.reannotation.columns:
        adata.obs[col] = alignment.reannotation[col]
    print(f"\n✓ Harmonized annotations transferred to adata.obs")

Cell 5: Visualize Harmonization Tree

In [ ]:
print("="*70)
print("Generating Harmonization Tree Plot")
print("="*70)

# Create figure for tree plot
fig, ax = plt.subplots(figsize=(16, 12), dpi=150)

# Generate tree plot
cellhint.treeplot(
    alignment,
    group_celltype=True,  # Group cell types by hierarchy
    order_dataset=False,  # Maintain alphabetical dataset order
    link_color='#0000007B',  # Semi-transparent black for links
    link_width=1.5,  # Link width
    node_shape='o',  # Circle nodes
    node_color=None,  # Auto-color matched cell types
    node_size=8.0,  # Node size
    show_label=True,  # Show cell type labels
    label_color='#000000',  # Black labels
    label_size=9,  # Label font size
    label_ha='center',  # Horizontal alignment
    label_va='top',  # Vertical alignment
    title=f'Cell Type Harmonization Tree\nAnnotation: {selected_ann_level}',
    ax=ax,
    figsize=None,  # Use provided figure
    show=False,  # Don't show yet
    save=False,  # Don't save yet
)

plt.tight_layout()
plt.savefig(
    fig_dir / 'cellhint_harmonization_tree.png',
    dpi=300,
    bbox_inches='tight',
    facecolor='white'
)
plt.show()

print(f"\n✓ Tree plot saved to: {fig_dir / 'cellhint_harmonization_tree.png'}")

Cell 6: Generate Sankey Plot for Harmonization

In [ ]:
print("="*70)
print("Generating Sankey Plot")
print("="*70)

try:
    # Create Sankey plot
    fig, ax = plt.subplots(figsize=(16, 10), dpi=150)
    
    cellhint.sankeyplot(
        alignment,
        ax=ax,
        title=f'Cell Type Harmonization Sankey\nAnnotation: {selected_ann_level}',
        show=False
    )
    
    plt.tight_layout()
    plt.savefig(
        fig_dir / 'cellhint_harmonization_sankey.png',
        dpi=300,
        bbox_inches='tight',
        facecolor='white'
    )
    plt.show()
    
    print(f"\n✓ Sankey plot saved to: {fig_dir / 'cellhint_harmonization_sankey.png'}")
    
except Exception as e:
    print(f"\nNote: Could not generate Sankey plot: {e}")
    print("This is optional and does not affect harmonization results")

Cell 7: Visualize Harmonized Annotations on UMAP

In [ ]:
print("="*70)
print("Visualizing Harmonized Annotations")
print("="*70)

# Get harmonized annotation column names
harm_cols = [col for col in adata.obs.columns if col.startswith('harm_')]

if harm_cols:
    print(f"\nFound {len(harm_cols)} harmonized annotation columns:")
    for col in harm_cols:
        print(f"  - {col}")
    
    # Plot original annotations
    fig, axes = plt.subplots(1, 2, figsize=(20, 8), dpi=150)
    
    # Original annotations
    sc.pl.umap(
        adata,
        color=selected_ann_level,
        title=f'Original Annotations\n({selected_ann_level})',
        ax=axes[0],
        show=False,
        frameon=False,
        legend_loc='right margin',
        legend_fontsize=8,
    )
    
    # Harmonized annotations (use the first harmonized column)
    harm_col = harm_cols[0]
    sc.pl.umap(
        adata,
        color=harm_col,
        title=f'Harmonized Annotations\n({harm_col})',
        ax=axes[1],
        show=False,
        frameon=False,
        legend_loc='right margin',
        legend_fontsize=8,
    )
    
    plt.tight_layout()
    plt.savefig(
        fig_dir / 'cellhint_harmonization_umap.png',
        dpi=300,
        bbox_inches='tight',
        facecolor='white'
    )
    plt.show()
    
    print(f"\n✓ UMAP comparison saved to: {fig_dir / 'cellhint_harmonization_umap.png'}")
    
    # Compare annotations per batch
    if len(batch_counts) <= 10:  # Only if reasonable number of batches
        fig, axes = plt.subplots(1, len(batch_counts), figsize=(6*len(batch_counts), 5), dpi=150)
        if len(batch_counts) == 1:
            axes = [axes]
        
        for idx, (batch, ax) in enumerate(zip(batch_counts.index, axes)):
            batch_mask = adata.obs[BATCH_KEY] == batch
            sc.pl.umap(
                adata[batch_mask, :],
                color=harm_col,
                title=f'{batch}\n({batch_mask.sum()} cells)',
                ax=ax,
                show=False,
                frameon=False,
                legend_loc='right margin',
                legend_fontsize=7,
            )
        
        plt.tight_layout()
        plt.savefig(
            fig_dir / 'cellhint_harmonization_by_batch.png',
            dpi=300,
            bbox_inches='tight',
            facecolor='white'
        )
        plt.show()
        
        print(f"✓ Per-batch UMAP saved to: {fig_dir / 'cellhint_harmonization_by_batch.png'}")
else:
    print("\nNo harmonized annotation columns found")


Cell 8: Export Harmonization Results (FIXED)

In [ ]:
print("="*70)
print("Exporting Harmonization Results")
print("="*70)

# Create harmonization output directory
harm_dir = output_dir / 'cellhint_harmonization'
harm_dir.mkdir(exist_ok=True)

# Export relation table
relation_path = harm_dir / 'harmonization_relation.csv'
alignment.relation.to_csv(relation_path)
print(f"✓ Relation table: {relation_path}")

# Export groups (handle different data types)
if hasattr(alignment, 'groups') and alignment.groups is not None:
    groups_path = harm_dir / 'harmonization_groups.csv'
    
    # Convert to DataFrame if it's a numpy array or Series
    if isinstance(alignment.groups, np.ndarray):
        # If it's a numpy array, save as a simple CSV
        groups_df = pd.DataFrame({'group': alignment.groups})
        groups_df.to_csv(groups_path, index=True)
        print(f"✓ Groups table: {groups_path} (converted from numpy array)")
    elif isinstance(alignment.groups, pd.Series):
        # If it's a Series, convert to DataFrame
        groups_df = alignment.groups.to_frame(name='group')
        groups_df.to_csv(groups_path)
        print(f"✓ Groups table: {groups_path} (converted from Series)")
    elif isinstance(alignment.groups, pd.DataFrame):
        # If it's already a DataFrame
        alignment.groups.to_csv(groups_path)
        print(f"✓ Groups table: {groups_path}")
    else:
        # For other types, try to convert to string representation
        with open(groups_path, 'w') as f:
            f.write(str(alignment.groups))
        print(f"✓ Groups saved as text: {groups_path}")

# Export reannotation
if hasattr(alignment, 'reannotation') and alignment.reannotation is not None:
    reannot_path = harm_dir / 'harmonization_reannotation.csv'
    alignment.reannotation.to_csv(reannot_path)
    print(f"✓ Reannotation table: {reannot_path}")

# Export base distance matrix
if hasattr(alignment, 'base_distance'):
    distance_path = harm_dir / 'base_distance_matrix.csv'
    
    # Handle base_distance matrix export
    try:
        if hasattr(alignment, 'base_distance_index') and hasattr(alignment, 'base_distance_columns'):
            # Use provided index and columns
            distance_df = pd.DataFrame(
                alignment.base_distance,
                index=alignment.base_distance_index,
                columns=alignment.base_distance_columns
            )
            distance_df.to_csv(distance_path)
        else:
            # Save as numpy array
            np.savetxt(distance_path, alignment.base_distance, delimiter=',')
        print(f"✓ Distance matrix: {distance_path}")
    except Exception as e:
        print(f"Note: Could not save distance matrix ({e})")

# Save alignment object (always works)
import pickle
alignment_path = harm_dir / 'alignment_object.pkl'
with open(alignment_path, 'wb') as f:
    pickle.dump(alignment, f)
print(f"✓ Alignment object: {alignment_path}")

print(f"\n✓ All harmonization results saved to: {harm_dir}")

Cell 9: Generate Harmonization Summary Report

In [ ]:
print("="*70)
print("Generating Harmonization Summary Report")
print("="*70)

# Create summary report
report = []
report.append("="*70)
report.append("CellHint Harmonization Analysis - Summary Report")
report.append("="*70)
report.append(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
report.append("")

report.append("[ Annotation Level ]")
report.append(f"  Selected level: {selected_ann_level}")
report.append(f"  Total cell types: {adata.obs[selected_ann_level].nunique()}")
report.append("")

report.append("[ Dataset Information ]")
report.append(f"  Batch key: {BATCH_KEY}")
report.append(f"  Number of batches: {len(batch_counts)}")
for batch, count in batch_counts.items():
    pct = count / adata.n_obs * 100
    report.append(f"    {batch}: {count:,} cells ({pct:.2f}%)")
report.append("")

report.append("[ Harmonization Parameters ]")
for key, value in CELLHINT_PARAMS.items():
    report.append(f"  {key}: {value}")
report.append("")

report.append("[ Harmonization Results ]")
report.append(f"  Relation table shape: {alignment.relation.shape}")
if hasattr(alignment, 'groups') and alignment.groups is not None:
    report.append(f"  Number of groups: {len(alignment.groups)}")
if hasattr(alignment, 'reannotation') and alignment.reannotation is not None:
    report.append(f"  Cells reannotated: {len(alignment.reannotation)}")
report.append("")

report.append("[ Output Files ]")
report.append(f"  Harmonization directory: {harm_dir}")
report.append(f"  - Relation table: harmonization_relation.csv")
report.append(f"  - Groups table: harmonization_groups.csv")
report.append(f"  - Reannotation table: harmonization_reannotation.csv")
report.append(f"  - Distance matrix: base_distance_matrix.csv")
report.append(f"  - Alignment object: alignment_object.pkl")
report.append("")

report.append("[ Visualization Files ]")
report.append(f"  - Tree plot: cellhint_harmonization_tree.png")
report.append(f"  - Sankey plot: cellhint_harmonization_sankey.png")
report.append(f"  - UMAP comparison: cellhint_harmonization_umap.png")
report.append(f"  - Per-batch UMAP: cellhint_harmonization_by_batch.png")
report.append("")

report.append("="*70)
report.append("Harmonization analysis completed successfully")
report.append("="*70)

report_text = '\n'.join(report)

# Save report
report_path = harm_dir / 'harmonization_summary.txt'
with open(report_path, 'w') as f:
    f.write(report_text)

print(f"\nSummary report saved to: {report_path}")
print("\n" + report_text)

Cell 10: Save Final Integrated Data with Harmonization

In [ ]:
print("="*70)
print("Saving Final Integrated Data")
print("="*70)

final_path = output_dir / 'adata_bbknn_cellhint_integrated.h5ad'

print(f"\nSaving integrated data with harmonization...")
print(f"  Cells: {adata.n_obs:,}")
print(f"  Genes: {adata.n_vars:,}")
print(f"  Using gzip compression (level 9)...")

adata.write_h5ad(final_path, compression='gzip', compression_opts=9)

file_size = final_path.stat().st_size / (1024**3)
print(f"\n✓ Saved: {final_path}")
print(f"  File size: {file_size:.2f} GB")

print("\n" + "="*70)
print("All CellHint harmonization steps completed successfully")
print("="*70)
print(f"\nKey outputs:")
print(f"  - Integrated data: {final_path}")
print(f"  - Harmonization results: {harm_dir}")
print(f"  - Visualizations: {fig_dir}")